# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

: 

In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Proszę Państwa, Wyspiański umiera
2. Szalone nożyczki (Teatr Bagatela)
3. Lustro. Spotkanie z Pawłem Danielem Zalewskim
4. Martwe natury
5. Nasze żony
6. Kino Bambino. Dzwieki miłości
7. Wróg ludu
8. Miserere Luminis, Detresse w Gwarku
9. Tradycyjne Święto Rękawki 2026
10. Piosenka jest dobra na wszystko

Czas wykonania (wątki): 1.09s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 4 procesach (rdzeniach)...
Zakończono w czasie 5.08s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [ ]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def download_site(fact):
    fact = requests.get(fact).json().get('fact')
    return fact

def run_sequential_demo():
    facts = [f"{CAT_API_URL}" for _ in range(20)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 20 faktów...")
    print(facts.__len__())
    start = time.time()
    
    all_facts = []
    for i in facts:
        all_facts.append(download_site(i))
        
    print(f"Pobrano łącznie {len(all_facts)} faktów.")
    print("Wyniki:")
    for i, fact in enumerate(all_facts[:20], 1):
        print(f"{i}. {fact}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")


def run_threaded_demo():
    facts = [f"{CAT_API_URL}" for _ in range(20)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 20 faktów...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        results = list(executor.map(download_site, facts))
    
    all_facts = [fact for fact in results]
    
    print(f"Pobrano łącznie {len(all_facts)} faktów.")
    print("Wyniki:")
    for i, fact in enumerate(all_facts[:20], 1):
        print(f"{i}. {fact}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

#run_sequential_demo()
run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 20 faktów...
Pobrano łącznie 20 faktów.
Wyniki:
1. Cat families usually play best in even numbers. Cats and kittens should be acquired in pairs whenever possible.
2. Cats respond better to women than to men, probably due to the fact that women's voices have a higher pitch.
3. The first cat show was in 1871 at the Crystal Palace in London.
4. In one stride, a cheetah can cover 23 to 26 feet (7 to 8 meters).
5. Two members of the cat family are distinct from all others: the clouded leopard and the cheetah. The clouded leopard does not roar like other big cats, nor does it groom or rest like small cats. The cheetah is unique because it is a running cat; all others are leaping cats. They are leaping cats because they slowly stalk their prey and then leap on it.
6. Cat's urine glows under a black light.
7. Cats should not be fed tuna exclusively, as it lacks taurine, an essential nutrient required for good feline health.  Make sure you have the proper Pet

### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [ ]:
import queue
import threading
import time

def producent(q, count):
    for i in range(count):
        q.put(i)
        print("Wyprodukowano: ", i)
        time.sleep(0.1)

    # sygnał zakończenia dla konsumentów
    q.put(None)
    q.put(None)

def konsument1(q):
    while True:
        i = q.get()

        if i is None:
            q.task_done()
            break

        if i % 2 == 0:
            print("Konsument 1 (parzyste): ", i)

        q.task_done()
        time.sleep(0.1)

def konsument2(q):
    while True:
        i = q.get()

        if i is None:
            q.task_done()
            break

        if i % 2 == 1:
            print("Konsument 2 (nieparzyste): ", i)

        q.task_done()
        time.sleep(0.1)

if __name__ == "__main__":
    q = queue.Queue()
    count = 20

    prod_thread = threading.Thread(target=producent, args=(q, count))
    cons_thread1 = threading.Thread(target=konsument1, args=(q,))
    cons_thread2 = threading.Thread(target=konsument2, args=(q,))

    prod_thread.start()
    cons_thread1.start()
    cons_thread2.start()

Wyprodukowano:  0
Konsument 1 (parzyste):  0


Wyprodukowano: Konsument 2 (nieparzyste):  1
 1
Wyprodukowano: Konsument 1 (parzyste):  2
 2
Wyprodukowano: Konsument 2 (nieparzyste):  3
 3
Wyprodukowano: Konsument 1 (parzyste):  4
 4
Wyprodukowano: Konsument 2 (nieparzyste):  5
 5
Wyprodukowano:  6
Wyprodukowano:  7
Wyprodukowano:  8
Wyprodukowano: Konsument 2 (nieparzyste):  9
 9
Wyprodukowano: Konsument 1 (parzyste):  10
 10
Wyprodukowano: Konsument 2 (nieparzyste):  11
 11
Wyprodukowano: Konsument 1 (parzyste):  12
 12
Wyprodukowano: Konsument 2 (nieparzyste):  13
 13
Wyprodukowano: Konsument 1 (parzyste):  14
 14
Wyprodukowano: Konsument 2 (nieparzyste):  15
 15
Wyprodukowano: Konsument 1 (parzyste):  16
 16
Wyprodukowano: Konsument 2 (nieparzyste):  17
 17
Wyprodukowano:  18
Wyprodukowano:  19


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [ ]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

def zliczanie_poteg():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_00000

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.map(calculate_power_sum, range(1, limit + 1))
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    zliczanie_poteg()   

Praca na 4 procesach (rdzeniach)...
Zakończono w czasie 8.84s.
